In [ ]:
# Code modified from function velocity_confidence from
# https://github.com/bowang-lab/DeepVelo/blob/main/deepvelo/utils/confidence.py

In [2]:
import sklearn
import scanpy as sc
import scvelo
import os
import pandas as pd
import numpy as np

In [3]:
import json
from typing import Dict
import torch
from pathlib import Path
from itertools import repeat
from collections import OrderedDict
from collections.abc import Mapping

In [4]:
from sklearn.metrics.pairwise import cosine_similarity
from scvelo import logging as logg
from scvelo.core import l2_norm, prod_sum
from scvelo.preprocessing.neighbors import get_neighs
from sklearn.preprocessing import normalize
#from deepvelo.plot.plot import statplot, compare_plot


In [5]:
def velocity_confidence(
    data,
    vkey="velocity",
    method="corr",
    scope_key=None,
    copy=False,
    only_velocity_genes=False,
    only_high_spearman=False,
):
    """Computes confidences of velocities.
    .. code:: python
        scv.tl.velocity_confidence(adata)
        scv.pl.scatter(adata, color='velocity_confidence', perc=[2,98])
    .. image:: https://user-images.githubusercontent.com/31883718/69626334-b6df5200-1048-11ea-9171-495845c5bc7a.png
       :width: 600px
    Arguments
    ---------
    data: :class:`~anndata.AnnData`
        Annotated data matrix.
    vkey: `str` (default: `'velocity'`)
        Name of velocity estimates to be used.
    method: `str` (default: `'corr'`), choice of `'corr'` or `'cosine'`
        Method to use for computing confidence, whether to use correlation or
        cosine similarity.
    scope_key: `str` (default: `None`)
        For each cell, cells with in scope_key are used for computing confidence.
        If `None`, use cell neighbors. Else, pick the cells in scope_key. Valid
        scope_key has to be in adata.obs.
    copy: `bool` (default: `False`)
        Return a copy instead of writing to adata.
    only_velocity_genes: `bool` (default: `False`)
        Only use velocity genes.
    only_high_spearman: `bool` (default: `False`)
        Only use high spearman.
    Returns
    -------
    velocity_length: `.obs`
        Length of the velocity vectors for each individual cell
    velocity_confidence: `.obs`
        Confidence for each cell
    """  # noqa E501

    adata = data.copy() if copy else data
    if vkey not in adata.layers.keys():
        raise ValueError("You need to run `tl.velocity` first.")
    if method not in ["corr", "cosine"]:
        raise ValueError("Method must be either 'corr' or 'cosine'.")
    if scope_key is not None and method == "corr":
        raise ValueError("Cannot use scope_key with method 'corr'.")

    V = np.array(adata.layers[vkey])

    # filter genes if needed
    #tmp_filter = np.invert(np.isnan(np.sum(V, axis=0)))
    if only_velocity_genes and (f"{vkey}_genes" in adata.var.keys()):
        tmp_filter &= np.array(adata.var[f"{vkey}_genes"], dtype=bool)
    if only_high_spearman and ("spearmans_score" in adata.var.keys()):
        tmp_filter &= adata.var["spearmans_score"].values > 0.1
    V = V #[:, tmp_filter]


    # zero mean, only need for correlation
    if method == "corr":
        V -= V.mean(1)[:, None]
    V_norm = l2_norm(V, axis=1)
    # normalize, only need for cosine similarity
    if method == "cosine":
        V /= V_norm[:, None]
    R = np.zeros(adata.n_obs)

    indices = (
        get_indices(dist=get_neighs(adata, "distances"))[0]
        if not scope_key
        else adata.obs[scope_key]  # the scope_key (e.g. cluster) of each cell
    )
    Vi_neighs_avg_cache = {}
    for i in range(adata.n_obs):
        if not scope_key:
            # use the neighbors of the cell
            Vi_neighs = V[indices[i]]
        else:
            # use the cells in scope_key
            if indices[i] not in Vi_neighs_avg_cache:
                Vi_neighs = V[indices == indices[i]]
                Vi_neighs_avg_cache[indices[i]] = Vi_neighs.mean(0)
        if method == "corr":
            Vi_neighs -= Vi_neighs.mean(1)[:, None]
            R[i] = np.mean(
                np.einsum("ij, j", Vi_neighs, V[i])
                / (l2_norm(Vi_neighs, axis=1) * V_norm[i])[None, :]
            )
        elif method == "cosine":
            # could compute mean first, because V has been normed
            Vi_neighs_avg = (
                Vi_neighs_avg_cache[indices[i]] if scope_key else Vi_neighs.mean(0)
            )
            R[i] = np.inner(V[i], Vi_neighs_avg)

    adata.obs[f"{vkey}_length"] = V_norm.round(2)
    adata.obs[f"{vkey}_confidence_{method}"] = R

    logg.hint(f"added '{vkey}_length' (adata.obs)")
    logg.hint(f"added '{vkey}_confidence_{method}' (adata.obs)")

    # if f"{vkey}_confidence_transition" not in adata.obs.keys():
    #     velocity_confidence_transition(adata, vkey)

    return adata if copy else None

Compare neighborhood consistency score

In [ ]:
##'pyro-velocity' 'cell2fate'
data_dir = '/data_path/benchmarking_results_revision/'
save_dir="/result_path/velocity_consistency_1st_batch/"
datasets=['Pancreas','DentateGyrus','Erythroid_Maturation','HumanBoneMarrow','Intestinal_organoid','mouse_retina','Hindbrain_GABA_Glio','organogenesis_chondrocyte']
vkey = "velocity"
method = "cosine"
scope_key = 'clusters'
methods=['pyro-velocity', 'cell2fate'] 
method_ind=np.arange(0, 2) #
data_ind=np.arange(0,8)#

In [ ]:
for j in method_ind:
    print(j)
    print(methods[j])
    for i in data_ind:
        adata= sc.read_h5ad(data_dir+methods[j]+"/CB_IC/"+f'{datasets[i]}_AnnData_Forscore.h5ad')
        velocity_confidence(adata, vkey, method, scope_key)  #
        if i==7 and j==0:
            print(f'{datasets[i]}\n{methods[j]}')
            print(f'{methods[j]}:{datasets[i]}')
            print(adata.obs['velocity_confidence_cosine'])
        adata.write_h5ad(save_dir+ datasets[i]+'/'+f'{datasets[i]}_{methods[j]}_consistency_score.h5ad')

In [ ]:
#'latentvelo'
data_dir = '/data_path/benchmarking_results_revision/'
save_dir="/result_path/velocity_consistency_1st_batch/"
datasets=['Pancreas','DentateGyrus','Erythroid_Maturation','HumanBoneMarrow','Intestinal_organoid','mouse_retina'] #'Hindbrain_GABA_Glio','organogenesis_chondrocyte'
method = "cosine"
vkey = "velocity"
scope_key = 'clusters'
methods=['latentvelo'] #
data_ind=np.arange(0,6)#
for i in data_ind:
    adata= sc.read_h5ad(data_dir+"/latentvelo/CB_IC/"+f'{datasets[i]}_AnnData_Forscore.h5ad')
    adata.layers['velocity']=adata.layers['spliced_velocity']
    velocity_confidence(adata, vkey, method, scope_key)  #
    if i==4:
        print(f'{datasets[i]}\nlatentvelo')
        print(f'lantentvelo:{datasets[i]}')
        print(adata.obs['velocity_confidence_cosine'])
    adata.write_h5ad(save_dir+ datasets[i]+'/'+f'{datasets[i]}_latentvelo_consistency_score.h5ad')

In [8]:
#Create a dataframe with the neighborhood consistency for downstream visualization(same datasets while different methods)
def within_cosineSim(adatalist=[], namelist=[]):
    n= len(namelist)
    mydfs=[]
    for i in range(n):
        adata = adatalist[i]
        name = namelist[i]
        mydf = adata.obs.copy()
        minidf = mydf[['clusters','velocity_confidence_cosine']]
        minidf = minidf.rename(columns={"velocity_confidence_cosine": name+'_'+'vel_confCos'})
        mydfs.append(minidf)

    big_df = pd.concat(mydfs, axis=1)
    # Drop duplicate columns
    big_df = big_df.loc[:, ~big_df.columns.duplicated()]

    return big_df

In [ ]:
        datadir="/~/velocity_consistency_1st_batch/"
        savepath="/~/velocity_consistency_1st_batch/value/"
        datasets=['Pancreas','DentateGyrus','Erythroid_Maturation','HumanBoneMarrow','Intestinal_organoid','mouse_retina','Hindbrain_GABA_Glio','organogenesis_chondrocyte']
        ind=np.arange(0,8)
        for i in ind:
                if  i==6 or i==7:
                        adata_pyro_velocity=sc.read_h5ad(datadir+datasets[i]+'/'+f'{datasets[i]}_pyro-velocity_consistency_score.h5ad')
                        adata_cell2fate=sc.read_h5ad(datadir+datasets[i]+'/'+f'{datasets[i]}_cell2fate_consistency_score.h5ad')
                        withinSim = within_cosineSim([adata_pyro_velocity,adata_cell2fate], ['pyro-velocity','cell2fate']) 
                        withinSim.to_csv(savepath+f'{datasets[i]}_neigh_consistency.csv')   
                else:
                        adata_pyro_velocity=sc.read_h5ad(datadir+datasets[i]+'/'+f'{datasets[i]}_pyro-velocity_consistency_score.h5ad')
                        adata_cell2fate=sc.read_h5ad(datadir+datasets[i]+'/'+f'{datasets[i]}_cell2fate_consistency_score.h5ad')
                        adata_latentvelo=sc.read_h5ad(datadir+datasets[i]+'/'+f'{datasets[i]}_latentvelo_consistency_score.h5ad')
                        withinSim = within_cosineSim([adata_pyro_velocity,adata_cell2fate, adata_latentvelo], ['pyro-velocity','cell2fate','latentvelo']) 
                        withinSim.to_csv(savepath+f'{datasets[i]}_neigh_consistency.csv')   
        print(datasets[i])
        print(withinSim.shape)
        print(withinSim.head(5))

In [ ]:
# Computed similarity within a method, also includes velocity signal computed for each 
import os
import numpy as np
import pandas as pd
os.chdir("/~/velocity_consistency_1st_batch/value/")
datasets=['Pancreas','DentateGyrus','Erythroid_Maturation','HumanBoneMarrow','Intestinal_organoid','mouse_retina','Hindbrain_GABA_Glio','organogenesis_chondrocyte']
index=np.arange(0,8)
for i in index:
    print(datasets[i])
    if i==0:
        df = pd.read_csv(f'{datasets[i]}_neigh_consistency.csv')
        df.rename(columns={df.columns[0]: 'index'}, inplace=True) #
        col_ind=[df.columns[1]]
        all_df_data = df_forBoxplot(df, list(col_ind),datasets[i], 'index')
    else:
        df = pd.read_csv(f'{datasets[i]}_neigh_consistency.csv')
        df.rename(columns={df.columns[0]: 'index'}, inplace=True) #
        col_ind=[df.columns[1]]
        df_data = df_forBoxplot(df, list(col_ind),datasets[i], 'index')
        all_df_data = pd.concat([all_df_data, df_data])
print(all_df_data.head())
print(all_df_data.shape)
all_df_data.to_csv('/save_path/all_datasets_neigh_consistency(3methods-1st_batch).csv')   